# Final GLM Model Status — Both Tasks, All 5 Rats (Notebooks 007-016 Complete)

**This is the complete status document for the GLM track of this project.** Both InSeq/OutSeq and Odor
Identity now have final, validated, statistically-tested models for all 5 rats. This notebook is
read-only, no new computation, it consolidates notebooks 014, 015, and 016 into one place using their
already-established, real numbers.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 1. Final Results, Both Tasks, All 5 Rats

| Rat | InSeq/OutSeq Acc | InSeq/OutSeq Window | Odor Identity Acc | Odor Identity Window | Both p < 0.005? |
|---|---|---|---|---|---|
| Mitt | 75.9% | 250ms @ +2350ms | 53.9% | 1750ms @ Poke-In | Yes |
| Barat | 79.8% | 250ms @ +1700ms | 53.0% | 1000ms @ Poke-In | Yes |
| Stella | 79.8% | 250ms @ +1500ms | 54.3% | 1750ms @ Poke-In | Yes |
| Superchris | 88.7% | 250ms @ +1000ms | 48.2% | 1500ms @ Poke-In | Yes |
| Buchanan | 71.2% | 250ms @ +1700ms | 59.9% | 1500ms @ Poke-In | Yes |

Chance levels: 50.0% (InSeq/OutSeq), 20.0% (Odor Identity). Every one of these 10 results (5 rats x 2
tasks) is a real, separately trained model, cross-validated, and confirmed statistically significant
via permutation testing.

**The two tasks use genuinely different windowing strategies, and that's a real finding, not an
inconsistency:** InSeq/OutSeq needs a short (250ms), precisely-timed window at a later, task-specific
offset. Odor Identity needs a much longer (1000-1750ms) window starting right at Poke-In. This makes
biological sense: the odor is present and constant from the start of the trial, so a model benefits from
integrating over more time, while the InSeq/OutSeq judgment appears to depend on a more specific,
later-occurring neural signature.


In [ ]:
rats = ['Mitt', 'Barat', 'Stella', 'Superchris', 'Buchanan']
inseq_final = [0.759, 0.798, 0.798, 0.887, 0.712]
odor_final = [0.539, 0.530, 0.543, 0.482, 0.599]

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(rats))
width = 0.35
bars1 = ax.bar(x - width/2, inseq_final, width, label='InSeq/OutSeq (chance=0.5)', color='#4C72B0')
bars2 = ax.bar(x + width/2, odor_final, width, label='Odor Identity (chance=0.2)', color='#DD8452')
ax.axhline(0.85, color='green', linestyle='--', alpha=0.6, label='InSeq/OutSeq target (~0.85)')
ax.set_xticks(x)
ax.set_xticklabels(rats)
ax.set_ylabel('Balanced accuracy')
ax.set_title('Final validated GLM models, both tasks, all 5 rats')
ax.legend()
for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.annotate(f'{h:.2f}', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 3),
                textcoords='offset points', ha='center', fontsize=9)
plt.tight_layout()
plt.show()


## 2. Interpretability (InSeq/OutSeq Only, From Notebook 015)

**High gamma is the most important frequency band overall** (mean |coefficient| 0.465), ranking above
theta (0.398), a direct validation of adding high gamma per the original advisor feedback. Theta remains
consistently the #2 band across every individual rat.

**Delta band results are unreliable** and excluded from interpretation: the 250ms InSeq/OutSeq window is
too short to resolve delta-range frequencies (1-4 Hz) properly, 2 of 5 rats showed exactly 0.0
importance, a resolution artifact, not a real finding.

**No channel is important across multiple rats**, each rat's top features involve entirely different
electrode numbers, expected given each animal has a physically distinct implant (confirmed via the
notebook 012 channel audit, which also found each rat is missing a different LFP channel).

**Not yet done: interpretability for Odor Identity's models.** Notebook 015 only covered InSeq/OutSeq.
Given Odor Identity uses an entirely different window (longer, starting at Poke-In), its important
bands/channels have not yet been examined and may differ from the InSeq/OutSeq pattern above.


## 3. How We Got Here: The Full Validation Arc (007-016)

1. **007**: Found and fixed a real sampling-rate bug (non-uniform recording rate breaking window
   duration assumptions).
2. **008**: Discovered the originally assumed fixed 500ms window was substantially suboptimal.
3. **009-011**: Validated a better window across all 5 rats through three rounds of increasing rigor
   (single-run → repeated cross-validation → extended per-rat safe span), catching and correcting
   search-noise artifacts along the way (e.g. implausible pre-Poke-In "best" windows).
4. **012**: Audited LFP channel structure across rats, found each rat missing a different channel
   (normal per-animal implant variation, not a data error), and fixed the pipeline to handle this
   automatically rather than assuming a fixed channel count.
5. **014**: Produced final, statistically-tested InSeq/OutSeq models for all 5 rats.
6. **015**: Interpretability analysis for those models, at the correct, validated windows.
7. **016**: Resolved Odor Identity's separate windowing problem (no single sharp peak existed; a longer
   continuous window won over both a short window and multi-window concatenation) and produced its
   final models.

This is a genuinely defensible chain: every number in Section 1's table traces back through a specific,
documented validation step, not an assumption.


## 4. What's NOT Done (Explicit, So Nothing Gets Overstated)

- **Odor Identity interpretability** (which bands/channels matter for that task specifically).
- **RNN retraining** at any of these validated windows. The RNN in notebook 04 still reflects the
  original, since-superseded fixed 500ms window and narrower band definitions. No fair comparison
  between the GLM and RNN currently exists.
- **Multi-rat pooled training.** Every result above is a separately trained, single-rat model. Pooling
  (with a session-based, leak-safe split) has not been attempted.
- **GNN**, the original stretch-goal architecture mentioned in early meeting notes, not started.

**Bottom line: the GLM track, for both tasks, across all 5 rats, is complete and defensible as a status
update.** The open items above are clearly separable next steps, not gaps in what's already been
reported.
